In [ ]:
import pandas as pd

# =====================================================
# 1. 기존 BERTopic 결과 불러오기
# =====================================================

INPUT_FILE = "arsenal_cafe_bertopic_result.xlsx"
OUTPUT_FILE = "arsenal_cafe_bertopic_macro_result.xlsx"

# 저장할 때 sheet_name을 다르게 했을 수 있으니 우선 시트 확인
xls = pd.ExcelFile(INPUT_FILE)
print(xls.sheet_names)

# 보통 topic_labeled_data 또는 cafe_topic_labeled_data에 원본+topic 결과가 있음
SHEET_NAME = "cafe_topic_labeled_data" if "cafe_topic_labeled_data" in xls.sheet_names else xls.sheet_names[0]

topic_df = pd.read_excel(INPUT_FILE, sheet_name=SHEET_NAME)

print(topic_df.shape)
print(topic_df.columns.tolist())

display(topic_df.head())

In [ ]:
# # =====================================================
# # 2. topic_name 없으면 생성
# # =====================================================

# if "topic_name" not in topic_df.columns:
#     if "Name" in topic_df.columns:
#         topic_df["topic_name"] = topic_df["Name"]
#     else:
#         topic_df["topic_name"] = topic_df["topic"].astype(str)

# print(topic_df[["topic", "topic_name"]].drop_duplicates().head(30))

In [ ]:
# =====================================================
# topic_info에서 Representation 붙이기
# =====================================================

topic_info = pd.read_excel(INPUT_FILE, sheet_name="topic_info")

rep_map = topic_info.set_index("Topic")["Representation"].to_dict()

topic_df["Representation"] = topic_df["topic"].map(rep_map)

display(
    topic_df[["topic", "topic_name", "Representation"]]
    .drop_duplicates()
    .sort_values("topic")
    .head(30)
)

In [ ]:
# =====================================================
# 세부 토픽 -> 상위 토픽 매핑: 키워드 점수 기반
# =====================================================

macro_keywords = {
    "시즌 성과/우승·챔스 반응": [
        "트로피", "우승", "우승컵", "무패", "메달", "잔류",
        "챔스", "챔결", "결승", "결승전", "클럽", "대항전",
        "pl", "시티", "동점", "연속", "세레모니", "물병",
        "순간", "벵거", "최고", "시간", "에너지", "기록",
        "레드", "카드", "승점", "마지막", "퍼레이드", "역대",
        "파리", "psg", "이강인"
    ],

    "이적시장/선수단 개편": [
        # 이적 행위/시장 키워드
        "이적", "시장", "영입", "계약", "주급", "판매", "가격",
        "이적료", "매물", "자원", "멀티", "임대", "관심",
        "포지션", "전문", "좌윙", "공미", "세컨", "가능",

        # 현재 아스날 소속이 아닌 외부 후보
        "바르콜라", "크루피", "토날리", "모건로저스", "고든",
        "스펜스", "스톤스", "아놀드", "케인", "토니", "벨링엄",
        "알바레즈", "디아라"
    ],

    "선수 개인 평가/폼·태도": [
        # 능력/태도 일반어
        "실력", "성실", "프로", "호감", "최애", "라커룸",
        "역할", "피지컬", "스타일", "슈팅", "미드필더",
        "왼쪽", "전진", "드리블", "패스", "주장", "리더",
        "부주장", "영향력", "건강", "마무리", "평화",
        "코너킥", "키커", "프리", "경합",

        # 아스날 선수단
        "마르티넬리", "가브리엘제주스", "가브리엘", "트로사르",
        "라이스", "외데고르", "화이트", "칼라피오리",
        "하베르츠", "팀버", "사카", "라야", "살리바",
        "메리노", "수비멘디", "루이스스켈리", "은와네리",
        "마두에케", "요케레스", "에제", "케파", "인카피에",
        "키비오르", "뇌가르", "모스케라", "셋포드"
    ],

    "감독/전술·구단 방향성": [
        "아르테타", "미켈", "테타", "감독", "전술", "라인업",
        "교체", "전략", "성향", "결단", "스승", "분위기",
        "미팅", "코치", "훈련", "성장", "시작", "자리"
    ],

    "팬 정서/응원·감사·낭만": [
        "행복", "친구", "기분", "낭만", "인생", "표정",
        "감사", "고생", "사랑", "자랑", "칭찬", "감동",
        "다행", "부탁", "더블", "평온", "성불", "꽃길",
        "화이팅", "goat", "월클", "잇츠"
    ],

    "구단·콘텐츠/기사·커뮤니티 노이즈": [
        "출처", "주요", "내용", "저작", "준수", "ig", "인스타",
        "공식", "프로젝트", "ai", "공신력", "친구", "실수",
        "추가", "gooners", "구너스", "와데고르", "no",
        "수정", "저장", "댓글", "알림", "악성", "공개",
        "포함", "일요일", "회의", "the", "of", "we", "in"
    ],

    "라이벌/외부팀 이슈": [
        "강등", "웨햄", "웨스트햄", "라이벌", "출신",
        "역전", "유나이티드", "시티", "맨시티", "첼시",
        "클롭", "토트넘", "케인"
    ],

    "기타/분류불가": [
        "스테이", "집안", "촬영", "부모", "간지"
    ]
}


def assign_macro_topic_score(topic_text):
    t = str(topic_text).lower()


    scores = {}

    for macro, keywords in macro_keywords.items():
        matched = [k for k in keywords if k in t]
        scores[macro] = len(matched)

    # 가장 많이 매칭된 카테고리 선택
    best_macro = max(scores, key=scores.get)
    best_score = scores[best_macro]

    # 아무 키워드도 안 걸리면 기타
    if best_score == 0:
        return "기타/분류불가"

    return best_macro

In [ ]:
topic_df["macro_topic"] = topic_df["topic_name"].apply(assign_macro_topic_score)

display(
    topic_df[["topic", "topic_name", "macro_topic"]]
    .drop_duplicates()
    .sort_values("topic")
)

In [ ]:
def get_macro_scores(topic_text):
    t = str(topic_text).lower()

    scores = {}

    for macro, keywords in macro_keywords.items():
        matched = [k for k in keywords if k in t]
        scores[macro] = {
            "score": len(matched),
            "matched_keywords": matched
        }

    return scores


# topic별 점수 확인용 테이블 만들기
score_rows = []

for _, row in (
    topic_df[["topic", "Representation"]]
    .drop_duplicates()
    .sort_values("topic")
    .iterrows()
):
    if row["topic"] == -1:
        macro_topic = "기타/분류불가"
        scores = {}
    else:
        scores = get_macro_scores(row["Representation"])
        macro_topic = assign_macro_topic_score(row["Representation"])

    score_rows.append({
        "topic": row["topic"],
        "Representation": row["Representation"],
        "macro_topic": assign_macro_topic_score(row["Representation"]),
        "scores": scores
    })

score_df = pd.DataFrame(score_rows)

display(score_df)

In [ ]:
# # =====================================================
# # 5. 필요하면 topic 번호 기준 수동 보정
# # =====================================================

# manual_override = {
#     # 예시: topic 번호: "상위 토픽명"
#     # 7: "선수 기량/포지션 평가",
#     # 8: "이적시장/선수보강",
#     # 10: "이적시장/선수보강",
#     # 11: "이적시장/선수보강",
#     # 18: "이적시장/선수보강",
#     # 22: "이적시장/선수보강",
# }

# for topic_id, macro in manual_override.items():
#     topic_df.loc[topic_df["topic"] == topic_id, "macro_topic"] = macro

# display(
#     topic_df[["topic", "topic_name", "macro_topic"]]
#     .drop_duplicates()
#     .sort_values("topic")
# )

In [ ]:
# =====================================================
# 6. 38R 전후 상위 토픽 비교
# =====================================================

period_count = pd.crosstab(
    topic_df["topic_period"],
    topic_df["macro_topic"]
)

period_ratio = pd.crosstab(
    topic_df["topic_period"],
    topic_df["macro_topic"],
    normalize="index"
) * 100

display(period_count)
display(period_ratio.round(2))

In [ ]:
# =====================================================
# 7. 긍정/부정별 상위 토픽 비교
# =====================================================

pn_df = topic_df[topic_df["gpt_sentiment"].isin(["positive", "negative"])].copy()

sentiment_count = pd.crosstab(
    pn_df["gpt_sentiment"],
    pn_df["macro_topic"]
)

sentiment_ratio = pd.crosstab(
    pn_df["gpt_sentiment"],
    pn_df["macro_topic"],
    normalize="index"
) * 100

display(sentiment_count)
display(sentiment_ratio.round(2))

In [ ]:
# =====================================================
# 8. 저장
# =====================================================

topic_macro_check = (
    topic_df
    .groupby(["topic", "topic_name", "macro_topic"])
    .size()
    .reset_index(name="count")
    .sort_values("topic")
)

with pd.ExcelWriter(OUTPUT_FILE) as writer:
    topic_df.to_excel(writer, sheet_name="topic_macro_data", index=False)
    topic_macro_check.to_excel(writer, sheet_name="topic_macro_check", index=False)
    period_count.to_excel(writer, sheet_name="period_macro_count")
    period_ratio.round(2).to_excel(writer, sheet_name="period_macro_ratio")
    sentiment_count.to_excel(writer, sheet_name="sentiment_macro_count")
    sentiment_ratio.round(2).to_excel(writer, sheet_name="sentiment_macro_ratio")

print("저장 완료:", OUTPUT_FILE)